# **LSTM for yf stocks dataset**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import yfinance as yf


class LSTM(nn.Module):
  def __init__(self,input_size,hidden_size,num_layers,output_size):
    super().__init__()
    self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
    self.fc = nn.Linear(hidden_size, output_size)

  def forward(self, x):
    out, _ = self.lstm(x)      # unpack tuple
    out = out[:, -1, :]        # last timestep
    out = self.fc(out)
    return out




In [ ]:


data = yf.download("AAPL", start="2008-01-01", end="2024-01-01")
print(data.head())
prices = data['Close'].values

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
prices_scaled = scaler.fit_transform(prices.reshape(-1,1))

X = prices_scaled[:-1]
y = prices_scaled[1:]

# Then create X_train, y_train, X_test, y_test from X and y

# X = prices[:-1]      # today
# y = prices[1:]       # tomorrow

n = int(0.8 * len(X))

X_train = X[:n]
y_train = y[:n]

X_test  = X[n:]
y_test  = y[n:]


X_train = torch.tensor(X_train).unsqueeze(-1).float()
y_train = torch.tensor(y_train).float()

X_test  = torch.tensor(X_test).unsqueeze(-1).float()
y_test  = torch.tensor(y_test).float()




In [ ]:
data.shape

In [ ]:
data[1:5]

In [ ]:
input_size = 1
hidden_size =512
num_layers = 2

lstm_model = LSTM(input_size, hidden_size, num_layers, output_size=1)
loss = nn.MSELoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)


In [ ]:
def lstm_train(model, X, y, loss_fn, optimizer, batch_size, epochs):
    model.train()
    n = X.size(0)

    for epoch in range(epochs):
        total_loss = 0

        for i in range(0, n, batch_size):
            optimizer.zero_grad()

            x_batch = X[i:i+batch_size]
            y_batch = y[i:i+batch_size]

            output = model(x_batch)
            loss = loss_fn(output, y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x_batch.size(0)

        print(f"Epoch {epoch+1} | Loss: {total_loss/n:.4f}")


In [ ]:
def evaluate_model(model, X, y, batch_size=64):
    model.eval()
    loss_fn = nn.MSELoss()

    total_loss = 0
    n = X.size(0)

    with torch.no_grad():
        for i in range(0, n, batch_size):
            x_batch = X[i:i+batch_size]
            y_batch = y[i:i+batch_size]

            output = model(x_batch)
            loss = loss_fn(output, y_batch)

            total_loss += loss.item() * x_batch.size(0)

    return total_loss / n


In [ ]:
lstm_losses = lstm_train(
    lstm_model,
    X_train,
    y_train,
    loss,
    optimizer,
    batch_size=64,
    epochs=15
)


In [ ]:
evaluate_model(lstm_model, X_test, y_test)

In [ ]:
import matplotlib.pyplot as plt

# Make predictions on test set
lstm_model.eval()
with torch.no_grad():
    y_pred = lstm_model(X_test).squeeze().numpy()

# If you scaled data, inverse transform
# y_pred = scaler.inverse_transform(y_pred.reshape(-1,1)).flatten()
# y_test_scaled = scaler.inverse_transform(y_test.reshape(-1,1)).flatten()

plt.figure(figsize=(12,6))
plt.plot(y_test.numpy(), label='Actual')
plt.plot(y_pred, label='Predicted')
plt.legend()
plt.title("Stock Price Prediction: Actual vs Predicted")
plt.show()


# **RNN**

In [ ]:
class RNN(nn.Module):
  def __init__(self,input_size,hidden_size,num_layers,output_size):
    super().__init__()
    self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
    self.fc = nn.Linear(hidden_size, output_size)

  def forward(self, x):
    out, _ = self.rnn(x)      # unpack tuple
    out = out[:, -1, :]        # last timestep
    out = self.fc(out)
    return out

In [ ]:
input_size = 5
hidden_size =256
num_layers = 4
rnn_model = RNN(input_size, hidden_size, num_layers, output_size=1)

In [ ]:
# def rnn_train(model, X, y, loss_fn, optimizer, batch_size, epochs):
#     model.train()
#     n = X.size(0)

#     for epoch in range(epochs):
#         total_loss = 0

#         for i in range(0, n, batch_size):
#             optimizer.zero_grad()

#             x_batch = X[i:i+batch_size]
#             y_batch = y[i:i+batch_size]

#             output = model(x_batch)
#             loss = loss_fn(output, y_batch)

#             loss.backward()
#             optimizer.step()

#             total_loss += loss.item() * x_batch.size(0)

#         print(f"Epoch {epoch+1} | Loss: {total_loss/n:.4f}")


In [ ]:
# def evaluate_model(model, X, y, batch_size=64):
#     model.eval()
#     loss_fn = nn.MSELoss()

#     total_loss = 0
#     n = X.size(0)

#     with torch.no_grad():
#         for i in range(0, n, batch_size):
#             x_batch = X[i:i+batch_size]
#             y_batch = y[i:i+batch_size]

#             output = model(x_batch)
#             loss = loss_fn(output, y_batch)

#             total_loss += loss.item() * x_batch.size(0)

#     return total_loss / n


In [ ]:
# rnn_losses = rnn_train(
#     rnn_model,
#     X_train,
#     y_train,
#     loss,
#     optimizer,
#     batch_size=64,
#     epochs=15
# )


In [ ]:
# evaluate_model(rnn_model, X_test, y_test)

In [ ]:
# import matplotlib.pyplot as plt

# # Make predictions on test set
# lstm_model.eval()
# with torch.no_grad():
#     y_pred = rnn_model(X_test).squeeze().numpy()

# # If you scaled data, inverse transform
# # y_pred = scaler.inverse_transform(y_pred.reshape(-1,1)).flatten()
# # y_test_scaled = scaler.inverse_transform(y_test.reshape(-1,1)).flatten()

# plt.figure(figsize=(12,6))
# plt.plot(y_test.numpy(), label='Actual')
# plt.plot(y_pred, label='Predicted')
# plt.legend()
# plt.title("Stock Price Prediction: Actual vs Predicted")
# plt.show()
